In [26]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.callbacks import Callback, LearningRateScheduler, EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

# Define dataset path & parameters
data_dir = "E:/Projects/Strabismus/Denoised"
img_height, img_width = 224, 224
batch_size = 64

num_classes = 5  

# Data augmentation & preprocessing
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.15,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_generator.classes)
class_weight_dict = dict(enumerate(class_weights))

# Load & modify VGG19 model
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
base_model.trainable = False  

# Define custom model
model = Sequential([
    base_model,
    Flatten(),
    Dense(1024, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.5),
    Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Define learning rate scheduler
def lr_scheduler(epoch, lr):
    return lr * 0.95 if epoch > 5 else lr

callback_lr = LearningRateScheduler(lr_scheduler)

# Handle optimizer compatibility
try:
    from tensorflow.keras.optimizers.legacy import AdamW
    optimizer = AdamW(learning_rate=0.001, weight_decay=1e-4)
except ImportError:
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Define custom metrics callback
class MetricsCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = model.predict(val_generator, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = val_generator.classes[:len(y_pred)]
        
        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=1)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=1)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=1)
        
        print(f'\nEpoch {epoch+1} Metrics:')
        print(f'Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')

# Define early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model (First Phase - Frozen Base Model)
model.fit(
    train_generator,
    epochs=60,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[MetricsCallback(), callback_lr, early_stopping]
)

# Unfreeze more layers for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-20]:  
    layer.trainable = False

# Recompile model with a lower learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Continue training with fine-tuning
model.fit(
    train_generator,
    epochs=40,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[MetricsCallback(), callback_lr, early_stopping]
)

# Implement Test-Time Augmentation (TTA) for inference
def tta_predict(model, image):
    aug_images = [datagen.random_transform(image) for _ in range(5)]
    preds = np.mean([model.predict(img[np.newaxis, :]) for img in aug_images], axis=0)
    return preds


Found 4395 images belonging to 5 classes.
Found 775 images belonging to 5 classes.
Epoch 1/60
69/69 [==============================] - ETA: 0s - loss: 4.1309 - accuracy: 0.2152
Epoch 1 Metrics:
Accuracy: 0.2194, Precision: 0.2133, Recall: 0.2203, F1 Score: 0.2047
69/69 [==============================] - 62s 823ms/step - loss: 4.1309 - accuracy: 0.2152 - val_loss: 2.4810 - val_accuracy: 0.2890 - lr: 0.0010
Epoch 2/60
69/69 [==============================] - ETA: 0s - loss: 2.3240 - accuracy: 0.2523
Epoch 2 Metrics:
Accuracy: 0.1987, Precision: 0.1714, Recall: 0.2003, F1 Score: 0.1375
69/69 [==============================] - 52s 757ms/step - loss: 2.3240 - accuracy: 0.2523 - val_loss: 2.1747 - val_accuracy: 0.3381 - lr: 0.0010
Epoch 3/60
69/69 [==============================] - ETA: 0s - loss: 2.0852 - accuracy: 0.2658
Epoch 3 Metrics:
Accuracy: 0.2065, Precision: 0.1884, Recall: 0.2011, F1 Score: 0.1593
69/69 [==============================] - 58s 837ms/step - loss: 2.0852 - accuracy: 0

ResourceExhaustedError: Graph execution error:

Detected at node 'sequential_8/vgg19/block3_conv3/Relu' defined at (most recent call last):
    File "E:\Anaconda\envs\matrix\lib\runpy.py", line 197, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "E:\Anaconda\envs\matrix\lib\runpy.py", line 87, in _run_code
      exec(code, run_globals)
    File "E:\Anaconda\envs\matrix\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "E:\Anaconda\envs\matrix\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "E:\Anaconda\envs\matrix\lib\site-packages\ipykernel\kernelapp.py", line 739, in start
      self.io_loop.start()
    File "E:\Anaconda\envs\matrix\lib\site-packages\tornado\platform\asyncio.py", line 205, in start
      self.asyncio_loop.run_forever()
    File "E:\Anaconda\envs\matrix\lib\asyncio\base_events.py", line 601, in run_forever
      self._run